# Synthetic Track Generation and Day-Night Merging Pipeline

## Overview
This document outlines the workflow for generating synthetic Clear Sky Index (CSI) estimation during nighttime with GUM Generator. By applying the GUM to nocturnal satellite bands and subsequently merging these estimation with daytime ground-truth CAMS data, the pipeline constructs a continuous and hourly time series.

---

## 1. Model Initialization and Data Loading
The pipeline initializes the pre-trained GUM model and prepares the global normalization metrics required for inference.

* **Model Checkpoint:** `Model_checkpoint/best_gum_temp_norm.pt`
* **Model Configuration:** `GUM(feature_size=64, K=2, gate_temp=0.7)`
* **Normalization Stats:** Loaded from `data/norm/minmax_stratified.npz` (calculated exclusively from daytime training data), and `hk_average_temperature.csv`.

## 2. Nighttime Feature Engineering
The raw nighttime satellite observations are aligned with static geographic and elevation maps to build the evaluation tensors.

* **Input Data:** `bands_night_paired_twillight.npz` or `bands_night_paired_twillight_with_elevation.npz` 
* **Static Maps:** `hk_geom_50x50.npz`
* **Bands Used:** 10 channels (`tbb_08` through `tbb_16`, plus `SAZ`)
* **Output Tensor:** `bands_night_pairs_with_elevation.npz`

## 3. Synthetic CSI Inference
The pipeline processes the prepared nighttime synthetic CSI through the GUM architecture in evaluation mode (`torch.no_grad()`) to generate synthetic CSI estimations. 

* **Batch Size:** 64
* **Output Export:** The synthetic nocturnal CSI arrays are saved sequentially to `csi_synthetic_night_temp_norm.npz`.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import torch
from noctis.gum.data_preprocessing import(
    build_band_tensor_with_elevation,
)
from noctis.gum.bands_and_cams import(
    load_bands_and_cams,
)
from noctis.gum.dataset import(
    BandsCamsDataset,
)
from torch.utils.data import DataLoader
from noctis.gum.utils import crop_back, pad_to_32, prepare_3ch_cams,clone_cams_15min_with_labels, qc_check_X, create_stale_baseline

In [ ]:
from pathlib import Path
import os
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "data").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent

# Change working directory to noctis/ root
os.chdir(ROOT_DIR)
print("Updated Working Directory:", os.getcwd())

Updated Working Directory: /Users/tjdu/Desktop/NOCITIS


## 1. Model Initialization and Data Loading

In [ ]:
from noctis.gum.model import GUM 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# torch.cuda.set_device(0)
model = GUM(feature_size=64, K=2, gate_temp=0.7).to(device)
PATH = "model_checkpoint/best_gum_temp_norm.pt"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
sd = torch.load(PATH, map_location=device)["model"]
if any(k.startswith("_orig_mod.") for k in sd.keys()):
    sd = {k.replace("_orig_mod.", ""): v for k, v in sd.items()}
model.load_state_dict(sd)
model.to(device)
print("Model loaded successfully.")

Model loaded successfully.


In [4]:
minmax =  np.load("data/norm/minmax_stratified.npz")
channel_min, channel_max = minmax["channel_min"], minmax["channel_max"]
# load night data with elevation and landmask
bands_npz_name = "data/gum/inference/bands_night_paired_twillight.npz"
elevation_npz_name = "data/elevation/hk_geom_50x50.npz"

## 2. Nighttime Feature Engineering

In [ ]:
bands_used = [
    'tbb_08','tbb_09','tbb_10','tbb_11','tbb_12',
    'tbb_13','tbb_14','tbb_15','tbb_16','SAZ'
]
shape, out = build_band_tensor_with_elevation(
    bands_npz_name=bands_npz_name,
    elevation_npz_name=elevation_npz_name,
    bands_used=bands_used,
    output_path="...",
)

In [5]:
X_night, y_night = load_bands_and_cams(
    bands_npz_path="data/gum/inference/bands_night_pairs_with_elevation.npz",
    cams_npz_path="data/gum/inference/cams_night_paired_twillight.npz",
    bands_indices_used=None,  
    eps=1e-6,
    verbose=False,
)

In [6]:
import pandas as pd

# additional temperature norm setup
temp_df = pd.read_csv("data/temperature/hk_average_temperature.csv")
temp_df['dt'] = pd.to_datetime(temp_df['Date time'].astype(str), format='%Y%m%d%H%M')
temp_df = temp_df.set_index('dt').sort_index()
temp_df['AverageTemperature'] = temp_df['AverageTemperature'].interpolate(method='time', limit_direction='both')
temp_df = temp_df.tz_localize('Asia/Hong_Kong').tz_convert('UTC')
temp_15_utc_k = temp_df['AverageTemperature'].resample('15min').interpolate(method='linear') + 273.15
cams = np.load("data/gum/inference/cams_night_paired_twillight.npz")
t = cams["time_min"]
t_dt_night = pd.to_datetime(t, unit='m', utc=True)
X_surface_k = temp_15_utc_k.reindex(t_dt_night, method='nearest').values

if np.isnan(X_surface_k).any():
    nan_count = np.isnan(X_surface_k).sum()
    print(f"Warning: Filled {nan_count} out-of-bounds samples with global mean.")
    X_surface_k = pd.Series(X_surface_k).fillna(np.nanmean(X_surface_k)).values

T_surf_broadcast = X_surface_k[:, np.newaxis, np.newaxis, np.newaxis]
if np.nanmax(X_night[:, :9, :, :]) < 5.0:
    print("⚠️ SKIPPED: Data is already normalized!")
    print("If you need to run this again, re-run the cell above that loads 'X' first.")
else:
    if X_night.dtype != np.float32:
        X_night = X_night.astype(np.float32)
    batch_size = 128  

    for i in range(0, X_night.shape[0], batch_size):
        batch_X = X_night[i : i + batch_size, :9, :, :]
        batch_T = T_surf_broadcast[i : i + batch_size]
        np.divide(batch_X, batch_T, out=batch_X)
        np.power(batch_X, 4, out=batch_X)

    print("Normalization complete! All CSV NaNs were healed before alignment.")
    print(f"Final check: Number of NaNs in X_night (first 9 channels): {np.isnan(X_night[:, :9, :, :]).sum()}")

Normalization complete! All CSV NaNs were healed before alignment.
Final check: Number of NaNs in X_night (first 9 channels): 0


In [ ]:
# Execute the QC
qc_check_X(X_test)

=== QC Report for Feature Tensor X ===

--- 1. Basic Properties ---
Shape: (59236, 12, 50, 50)
Data Type: float32

--- 2. Mathematical Integrity ---
Total NaNs: 0
Total Infs: 0
✅ PASS: X is completely clean of NaNs and infinite values.

--- 3. Normalized TBB Channels (0-8) ---
Global Min:  0.1368  (Expected: ~0.1 to 0.4 for very cold clouds)
Global Max:  1.1540  (Expected: ~1.0 to 1.5 for hot ground)
Global Mean: 0.6601
✅ PASS: (T_sat / T_surface)^4 normalization range looks physically sound.

--- 4. Auxiliary Geo-Channels (9-11) ---
Global Min:  0.0000
Global Max:  804.1716
Global Mean: 24.7169


In [7]:
night_dataset  = BandsCamsDataset(X_night,  y_night,  channel_min, channel_max,
                                 elev_ch_idx=-2, land_ch_idx=-1)
night_loader  = DataLoader(night_dataset,  batch_size=64, shuffle=False)

## 3. Synthetic CSI Inference

In [ ]:
# save synthetic night CSI (time-consuming)
y_pred = []
model.eval()
with torch.no_grad():
    for x_batch, y_batch in night_loader:
        x_batch = x_batch.to(device)
        xb_pad, h, w = pad_to_32(x_batch)

        pred_y = model(xb_pad)
        if isinstance(pred_y, (tuple, list)):
            pred_y = pred_y[0]
            pred_y = crop_back(pred_y, h, w).contiguous()
            y_pred.append(pred_y.cpu().numpy())
y_pred = np.concatenate(y_pred, axis=0)

out_path = "data/gum/synthetic/csi_synthetic_night_temp_norm.npz"
night_t    = np.load(bands_npz_name)["time_min"]
np.savez_compressed(
    out_path,
    csi=y_pred,
    time_hourly=night_t,
)

In [ ]:
path_cams_raw_15min = "data/cams/ghi_grid2500_2ch_2021_2023.npz"
path_cams_3ch_baseline = "data/cams/ghi_grid2500_3ch_2021_2023_baseline_15mins.npz"

prepare_3ch_cams(
    in_path=path_cams_raw_15min,
    out_path=path_cams_3ch_baseline,
    clip_csi=(0.0, 1.0) 
)

In [ ]:
# Overwrite Channel 2 with the synthetic CSI frames using Strategy B
clone_cams_15min_with_labels(
    path_syn= "data/gum/synthetic/csi_synthetic_night_temp_norm.npz",
    path_cams_15min= "data/cams/ghi_grid2500_3ch_2021_2023_baseline_15mins.npz",
    out_path="data/gum/synthetic/15mins_synthetic_label_test.npz",
    csi_channel=2,
    missing_label=-1.0
)

In [4]:
# for forecasting engines
syn = np.load("data/gum/synthetic/15mins_synthetic_label.npz")
syn["data"].shape

(105078, 3, 50, 50)

In [5]:
# night csi all 0
raw_A = np.load("data/cams/ghi_grid2500_3ch_2021_2023_baseline_15mins.npz")
print(raw_A["data"].shape)

input_file = 'Data/CAMS/ghi_grid2500_3ch_2021_2023_baseline_15mins.npz'
output_file = 'Data/CAMS/ghi_grid2500_3ch_2021_2023_baseline_15mins_robust(stale).npz'
# night csi adapt from last available dusk CSI
create_stale_baseline(input_file, output_file, safe_threshold=5.0)

# night csi adapt from last available dusk CSI
raw_A = np.load("data/cams/ghi_grid2500_3ch_2021_2023_baseline_15mins_robust(stale)_csi.npz")
print(raw_A["data"].shape)

(105078, 3, 50, 50)
(105078, 3, 50, 50)


In [3]:
raw_A = np.load("data/cams/ghi_grid2500_3ch_2021_2023_baseline_15mins_robust(stale)_csi.npz")
print(raw_A["data"].shape)

(105078, 3, 50, 50)
